# Input-Adaptive Teacher-to-Student Handover Scheduling
## High-Speed GPU Benchmark & Demonstration (Google Colab / Colab Pro)

**Author:** Rhenel Jhon Sajol (Master's Thesis 2026)  
**Hardware Target:** NVIDIA CUDA GPU (A100, V100, T4)  
**Repository:** [Shonget-Thesis/Shonget-DMOSpeech2](https://github.com/Shonget-Thesis/Shonget-DMOSpeech2)

---
### Overview
This notebook runs the **Input-Adaptive Teacher-to-Student Handover Scheduler** on an NVIDIA GPU via Google Colab. 
On GPU, execution runs **~100x faster** than on laptop CPU (~0.25s per utterance vs ~9.08s on CPU), enabling full held-out benchmark runs in minutes.

### Step 1: Verify NVIDIA GPU Allocation
Make sure your Colab runtime is set to **GPU** (`Runtime > Change runtime type > T4 GPU or A100 GPU`).

In [ ]:
!nvidia-smi

### Step 2: Clone Repository & Install Dependencies

In [ ]:
# Clone the thesis repository
!git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git
%cd Shonget-DMOSpeech2

# Install core audio and ML evaluation packages
!pip install -q vocos speechbrain transformers jiwer soundfile librosa pandas pyarrow scipy jieba

### Step 3: Download DMOSpeech 2 Checkpoints
Downloads `model_85000.pt` (4.0 GB) and `model_1500.pt` (270 MB) directly from HuggingFace into `ckpts/`.

In [ ]:
!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

### Step 4: Seed-TTS Dataset Setup
Clone the official Seed-TTS test dataset evaluation repository to get test prompts:

In [ ]:
# Download official Seed-TTS evaluation dataset
!git clone https://github.com/BytedanceSpeech/seed-tts-eval.git data/seed-tts-eval
!ls -la data/seed-tts-eval/en

### Step 5: Run High-Speed GPU Benchmark
You can select the scope:
- `--n_speakers 10` (Quick smoke test, finishes in ~20 seconds)
- `--scope test200` (Full 200 held-out test speakers, finishes in ~5 minutes on A100/T4!)
- `--scope expanded` (All speakers in the dataset)

In [ ]:
# Execute GPU benchmark
!python "Handover Scheduling/colab/run_gpu_expanded_benchmark.py" --device cuda --scope test200 --n_speakers 10

### Step 6: Statistical Analysis & Policy Comparison
Evaluates the Proposed Adaptive Policy vs. Baselines on the generated GPU results.

In [ ]:
import pandas as pd
import numpy as np

csv_path = "Handover Scheduling/colab/results/gpu_grid_test200.csv"
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} evaluations across {df['speaker_id'].nunique()} speakers!")

# Display summary table
summary = df.groupby(["teacher_step", "condition"])[["nfe", "wer", "similarity", "utmos", "is_failure", "synthesis_time", "rtf"]].mean()
display(summary)

### Step 7: Interactive Audio Player
Listen to speech samples synthesized on your GPU directly in the notebook cells:

In [ ]:
import IPython.display as ipd
import glob

audio_files = glob.glob("Handover Scheduling/demo/audio/*.wav")
print(f"Found {len(audio_files)} demo audio files:")

for f in sorted(audio_files):
    print("\n" + "="*60)
    print(f"File: {f}")
    ipd.display(ipd.Audio(f))